# 🧃 بدّلها لطفلي — Notebook 1: جمع البيانات والتحقق من جودتها وتنظيفها (PySpark)

**الهدف:** نحمّل قاعدة Open Food Facts كاملة، ونفلتر منتجات الخليج بـ PySpark، ونقيس جودة البيانات، وننظفها، ونصنّف المنتجات اللي يستهلكها الأطفال، ونقيس دقة التصنيف، ونطلع أول أرقام عن السكر.

**أهم التحسينات (النسخة الثالثة):**
1. **تشغيل ذكي:** إذا طبقة Bronze موجودة من قبل، نتخطى التحميل والقراءة الثقيلة (من 20 دقيقة لدقيقة).
2. **قواعد تصنيف أدق:** تنظيف الوسوم، والشوكولاتة قبل الحلويات، واستبعاد خبز البرجر وأقماع الآيس كريم الفاضية والبطاطس المقلية.
3. **مجموعة تقييم ثابتة (Ground Truth)** من مراجعتين يدويتين (247 منتج)، نختبر عليها أي تعديل على القواعد تلقائياً.
4. **عينة Holdout نهائية** ما شفناها أبداً وقت تعديل القواعد، عشان رقم الدقة يكون صادق.
5. **قيم الملح الشاذة:** أي ملح أعلى من 5 جرام لكل 100 جرام نعتبره خطأ إدخال ونستبعده من الإحصائيات.

**التشغيل:** شغّلي الخلايا بالترتيب (Shift + Enter).

## 1) ربط Google Drive وتجهيز المجلدات

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, shutil
PROJECT_DIR = '/content/drive/MyDrive/kids_food_swap'
BRONZE  = f'{PROJECT_DIR}/bronze'    # البيانات بعد الفلترة الأولى (قريبة من الخام)
SILVER  = f'{PROJECT_DIR}/silver'    # البيانات النظيفة
GOLD    = f'{PROJECT_DIR}/gold'      # الجاهزة للتحليل
REPORTS = f'{PROJECT_DIR}/reports'   # تقارير الجودة والمراجعة
for d in [BRONZE, SILVER, GOLD, REPORTS]:
    os.makedirs(d, exist_ok=True)

BRONZE_PATH = f'{BRONZE}/gulf_products.parquet'
REBUILD = False   # خليها True إذا تبين تعيدين بناء Bronze من البيانات الخام (مثلاً بعد تحديث Open Food Facts)
BRONZE_READY = os.path.exists(f'{BRONZE_PATH}/_SUCCESS') and not REBUILD
print('مجلد المشروع:', PROJECT_DIR)
print('Bronze جاهز من قبل؟', BRONZE_READY, '(إذا True نتخطى التحميل والقراءة الثقيلة)')

## 2) تثبيت PySpark

In [ ]:
!pip install -q pyspark

## 3) تحميل قاعدة Open Food Facts (تقريباً 1GB مضغوط)
`_SUCCESS` ملف صغير يكتبه Spark لما يخلص الحفظ بنجاح. وجوده يعني إن Bronze مكتمل، فما نحتاج نحمّل من جديد.

In [ ]:
import subprocess
RAW = '/content/off.csv.gz'
if BRONZE_READY:
    print('⏭️ Bronze موجود، ما نحتاج نحمّل البيانات الخام')
elif not os.path.exists(RAW):
    subprocess.run(['wget', '-q', '-O', RAW,
                    'https://static.openfoodfacts.org/data/en.openfoodfacts.org.products.csv.gz'], check=True)
    print('✅ تم التحميل:', round(os.path.getsize(RAW) / 1e9, 2), 'GB')

## 4) تشغيل Spark

In [ ]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
import pandas as pd

spark = (SparkSession.builder
         .appName('KidsFoodSwap')
         .config('spark.driver.memory', '10g')
         .config('spark.sql.shuffle.partitions', '16')
         .getOrCreate())
spark

## 5) قراءة البيانات واختيار الأعمدة المهمة

In [ ]:
if not BRONZE_READY:
    raw = (spark.read
           .option('header', True)
           .option('sep', '\t')
           .option('quote', '')
           .option('mode', 'DROPMALFORMED')
           .csv(RAW))

    wanted = ['code', 'product_name', 'brands', 'countries_tags', 'categories_tags',
              'serving_size', 'nutriscore_grade', 'nova_group', 'additives_n',
              'energy-kcal_100g', 'sugars_100g', 'salt_100g', 'fat_100g',
              'saturated-fat_100g', 'fiber_100g', 'proteins_100g',
              'last_modified_t', 'image_small_url']
    cols = [c for c in wanted if c in raw.columns]
    print('أعمدة مطلوبة غير موجودة:', [c for c in wanted if c not in raw.columns])

    df = raw.select(*cols)
    print('عدد الأعمدة الأصلية:', len(raw.columns), '| اخترنا:', len(cols))
else:
    print('⏭️ نتخطى القراءة: Bronze جاهز')

## 6) فلترة منتجات الخليج وحفظها في طبقة Bronze 🇸🇦
أثقل خطوة، لأن Spark يمر على ملايين الصفوف.

In [ ]:
gulf = ['saudi-arabia', 'united-arab-emirates', 'kuwait', 'qatar', 'bahrain', 'oman']
pattern = '|'.join([f'en:{c}' for c in gulf])

if not BRONZE_READY:
    df.filter(F.col('countries_tags').rlike(pattern)).write.mode('overwrite').parquet(BRONZE_PATH)

gulf_df = spark.read.parquet(BRONZE_PATH)
print('عدد منتجات الخليج:', gulf_df.count())

> 💡 ما تحتاجين تتخطين خلايا يدوياً بعد الحين: الكود يتخطى التحميل والقراءة الثقيلة لحاله إذا Bronze موجود.

In [ ]:
# كم منتج في كل دولة؟
(gulf_df
 .withColumn('country', F.explode(F.split('countries_tags', ',')))
 .filter(F.col('country').rlike(pattern))
 .groupBy('country').count()
 .orderBy(F.desc('count'))
 .show())

## 7) تحويل أنواع الأعمدة
كل الأعمدة انقرت كنص. نحول الأرقام لأرقام، وننظف المسافات من النصوص.

In [ ]:
num_cols = ['energy-kcal_100g', 'sugars_100g', 'salt_100g', 'fat_100g', 'saturated-fat_100g',
            'fiber_100g', 'proteins_100g', 'additives_n', 'nova_group', 'last_modified_t']

typed = gulf_df
for c in num_cols:
    if c in typed.columns:
        typed = typed.withColumn(c, F.col(c).cast('double'))

typed = (typed
         .withColumn('code', F.trim('code'))
         .withColumn('product_name', F.trim('product_name'))
         .withColumn('nutriscore_grade', F.lower(F.trim('nutriscore_grade'))))

def is_blank(c):
    """True إذا العمود النصي فاضي أو كله مسافات."""
    return F.col(c).isNull() | (F.col(c) == '')

## 8) تقرير جودة البيانات (Data Quality Report) 🩺
قبل ما نحذف أي شي، نقيس المشاكل ونوثقها.

In [ ]:
q = typed.agg(
    F.count('*').alias('total_rows'),
    F.sum(is_blank('code').cast('int')).alias('missing_barcode'),
    F.sum(is_blank('product_name').cast('int')).alias('missing_name'),
    F.sum(F.col('sugars_100g').isNull().cast('int')).alias('missing_sugar'),
    F.sum((~F.col('sugars_100g').between(0, 100)).cast('int')).alias('sugar_out_of_range'),
    F.sum(F.col('salt_100g').isNull().cast('int')).alias('missing_salt'),
    F.sum((~F.col('salt_100g').between(0, 100)).cast('int')).alias('salt_out_of_range'),
    F.sum(F.col('nutriscore_grade').isin('a', 'b', 'c', 'd', 'e').cast('int')).alias('has_nutriscore'),
).first().asDict()

dups = typed.filter(~is_blank('code')).groupBy('code').count().filter('count > 1')
q['duplicated_barcodes'] = dups.count()
q['extra_duplicate_rows'] = dups.agg(F.sum(F.col('count') - 1)).first()[0] or 0

total = q['total_rows']
quality_report = pd.DataFrame(
    [(k, int(v or 0), round(100 * (v or 0) / total, 1)) for k, v in q.items()],
    columns=['metric', 'count', 'pct_of_rows'])
quality_report.to_csv(f'{REPORTS}/data_quality_report.csv', index=False)
quality_report

## 9) التنظيف وإزالة التكرار 🧹
كل خطوة نسجل كم صف حذفت، عشان تعرفين وين راحت البيانات.

In [ ]:
steps = []
def track(name, d):
    steps.append((name, d.count()))
    return d

d = track('0. البداية', typed)
d = track('1. حذف المنتجات بدون باركود', d.filter(~is_blank('code')))
d = track('2. حذف المنتجات بدون اسم', d.filter(~is_blank('product_name')))
d = track('3. حذف المنتجات بدون قيمة سكر', d.filter(F.col('sugars_100g').isNotNull()))
d = track('4. حذف السكر خارج 0–100', d.filter(F.col('sugars_100g').between(0, 100)))
d = track('5. حذف الملح خارج 0–100', d.filter(F.col('salt_100g').isNull() | F.col('salt_100g').between(0, 100)))

# درجة الاكتمال: كم عمود مهم فيه قيمة
text_keys = [c for c in ['brands', 'categories_tags', 'nutriscore_grade', 'serving_size'] if c in d.columns]
num_keys  = [c for c in ['energy-kcal_100g', 'salt_100g', 'fat_100g', 'saturated-fat_100g',
                         'fiber_100g', 'proteins_100g'] if c in d.columns]
completeness = (sum(F.when(~is_blank(c), 1).otherwise(0) for c in text_keys) +
                sum(F.when(F.col(c).isNotNull(), 1).otherwise(0) for c in num_keys))

# من كل باركود مكرر: نخلي الأكثر اكتمالاً، ثم الأحدث تعديلاً
order = [F.desc('completeness_score')]
if 'last_modified_t' in d.columns:
    order.append(F.desc_nulls_last('last_modified_t'))
order.append(F.asc('product_name'))   # كسر التعادل عشان النتيجة تكون ثابتة كل مرة
w = Window.partitionBy('code').orderBy(*order)

d = (d.withColumn('completeness_score', completeness)
      .withColumn('rn', F.row_number().over(w))
      .filter(F.col('rn') == 1)
      .drop('rn'))
d = track('6. إزالة التكرار (الأكثر اكتمالاً)', d)

cleaning_log = pd.DataFrame(steps, columns=['step', 'rows'])
cleaning_log['removed'] = (cleaning_log['rows'].shift(1) - cleaning_log['rows']).fillna(0).astype(int)
cleaning_log.to_csv(f'{REPORTS}/cleaning_log.csv', index=False)

SILVER_PATH = f'{SILVER}/products_clean.parquet'
d.write.mode('overwrite').parquet(SILVER_PATH)
clean = spark.read.parquet(SILVER_PATH)
cleaning_log

## 10) تصنيف المنتجات 👶
نفصل بين نوعين:
- **Infants (targeted):** أغذية الرضع، مخصصة للأطفال فعلاً.
- **Commonly eaten by kids:** سناكات ومشروبات عامة يستهلكها الأطفال غالباً.

**كيف تشتغل القواعد:**
| الجزء | وش يسوي | ليش |
| --- | --- | --- |
| تنظيف الوسوم | حروف صغيرة، والمسافة تصير شرطة | بعض الوسوم مكتوبة يدوياً مثل `en:Yoghurt drink`، فنوحدها لـ `en:yoghurt-drink` |
| `arrays_overlap` | يطابق **الوسم كامل** | البحث بجزء من الكلمة كان يخلي `cakes` تطابق `puffed-rice-cakes` |
| قائمة الاستبعاد | ترفض المنتج حتى لو انطبق الشرط | مثلاً خبز البرجر عنده وسم `brioches` بس هو خبز مو معجنات |
| الترتيب | أول قاعدة تنطبق تُعتمد | الأشكال المحددة (بسكويت، كيك) قبل المكونات (شوكولاتة)، والشوكولاتة قبل الحلويات |

**التعديلات في v3** (من أخطاء عينة الاختبار):
- الشوكولاتة صارت **قبل** الحلويات، مع استبعاد البونبون وألواح الحلوى (`candy-chocolate-bars`)، عشان لوح شوكولاتة بالمكسرات ينحسب شوكولاتة.
- استبعاد `hamburger-buns` و`special-breads` من المعجنات.
- استبعاد `cone-wafer-for-ice-cream` (الأقماع الفاضية) من الآيس كريم.
- استبعاد `potato-fries` من الشيبس.

In [ ]:
tags = F.split(F.regexp_replace(F.lower(F.coalesce(F.col('categories_tags'), F.lit(''))), r'\s+', '-'), ',')
name = F.lower(F.coalesce(F.col('product_name'), F.lit('')))

def has_any(tag_list):
    """True إذا المنتج عنده وسم واحد على الأقل من القائمة (مطابقة كاملة)."""
    return F.arrays_overlap(tags, F.array(*[F.lit(t) for t in tag_list]))

rules = [  # (الفئة, وسوم تدخله, وسوم تستبعده, نمط في اسم المنتج)
    ('Baby food',         ['en:baby-foods', 'en:baby-milks', 'en:infant-formulas'], [], None),
    ('Yogurts',           ['en:yogurts', 'en:yogurt-drinks', 'en:drinkable-yogurts', 'en:fermented-milk-drinks',
                           'en:kefir', 'en:yoghurts', 'en:yoghurt-drink'], ['en:puddings'], None),
    ('Flavoured milk',    ['en:flavoured-milks', 'en:chocolate-milks'], [], None),
    ('Ice cream',         ['en:ice-creams', 'en:ice-creams-and-sorbets', 'en:ice-cream-bars'],
                          ['en:cone-wafer-for-ice-cream'], None),
    ('Juices',            ['en:juices-and-nectars', 'en:fruit-juices', 'en:fruit-nectars'], [], None),
    ('Soft drinks',       ['en:sodas', 'en:soft-drinks', 'en:colas'], [], None),
    ('Breakfast cereals', ['en:breakfast-cereals'], [], None),
    ('Cakes & pastries',  ['en:cakes', 'en:muffins', 'en:croissants', 'en:viennoiseries',
                           'en:sweet-pastries-and-pies', 'en:pastries', 'en:brioches'],
                          ['en:puffed-cereal-cakes', 'en:cooking-helpers', 'en:baking-mixes', 'en:pastry-helpers',
                           'en:hamburger-buns', 'en:special-breads'], None),
    ('Biscuits & wafers', ['en:biscuits', 'en:cookies', 'en:wafers', 'en:chocolate-biscuits'],
                          ['en:crackers', 'en:crackers-appetizers'], r'wafer'),
    ('Chocolate',         ['en:chocolates', 'en:milk-chocolates', 'en:dark-chocolates', 'en:white-chocolates',
                           'en:chocolate-spreads'],
                          ['en:cocoa-and-chocolate-powders', 'en:instant-beverages', 'en:bonbons',
                           'en:candy-chocolate-bars'], None),
    ('Candies',           ['en:candies', 'en:confectioneries', 'en:chocolate-candies',
                           'en:gummi-candies', 'en:lollipops', 'en:bonbons'], [], None),
    ('Chips',             ['en:crisps', 'en:chips-and-fries', 'en:potato-crisps', 'en:tortilla-chips'],
                          ['en:potato-fries', 'en:fries'], None),
]

def rule_condition(include, exclude, name_pat):
    cond = has_any(include)
    if name_pat:
        cond = cond | name.rlike(name_pat)
    if exclude:
        cond = cond & ~has_any(exclude)
    return cond

conditions = [(g, rule_condition(inc, exc, pat)) for g, inc, exc, pat in rules]
group_expr = F.when(conditions[0][1], conditions[0][0])
for g, cond in conditions[1:]:
    group_expr = group_expr.when(cond, g)
n_matches = sum(F.when(cond, 1).otherwise(0) for _, cond in conditions)

kids = (clean
        .withColumn('product_group', group_expr)
        .filter(F.col('product_group').isNotNull())
        .withColumn('audience', F.when(F.col('product_group') == 'Baby food', 'Infants (targeted)')
                                 .otherwise('Commonly eaten by kids'))
        .withColumn('n_rule_matches', n_matches)
        .withColumn('sugar_tsp_100g', F.round(F.col('sugars_100g') / 4, 1)))   # ملعقة صغيرة ≈ 4g سكر

print('عدد المنتجات المصنفة:', kids.count())
print('منها ينطبق عليها أكثر من فئة:', kids.filter('n_rule_matches > 1').count())
kids.groupBy('audience', 'product_group').count().orderBy('audience', F.desc('count')).show(truncate=False)

## 11) قياس دقة التصنيف 🎯
### أ) مجموعة التقييم الثابتة (Ground Truth)
راجعنا يدوياً عينتين قبل (124 + 123 منتج). هنا نحولها لـ **"الإجابة الصحيحة" لكل منتج**:
- إذا التصنيف القديم كان صح، الفئة الصحيحة = نفس الفئة.
- إذا كان غلط، نكتب الفئة الصحيحة في `CORRECTIONS`، أو `None` إذا المنتج ما ينتمي لأي فئة (مثل بودرة الكاكاو).

بعدها نقدر نختبر **أي قواعد جديدة** تلقائياً على 247 منتج بدون مراجعة يدوية جديدة. هذا مثل **Unit Tests** للتصنيف.

⚠️ **ملاحظة صادقة:** عدلنا القواعد بناءً على أخطاء العينتين، فالدقة عليهم **متفائلة**. الرقم الحقيقي يجي من عينة Holdout في الخطوة (ب).

In [ ]:
import numpy as np

DEV_PATH  = f'{REPORTS}/classification_review.csv'   # المراجعة الأولى (seed=42)
TEST_PATH = f'{REPORTS}/classification_test.csv'     # المراجعة الثانية (seed=7)
GT_PATH   = f'{REPORTS}/classification_ground_truth.csv'

CORRECTIONS = {   # باركود (بدون أصفار البداية) -> الفئة الصحيحة. أي منتج غلط مو موجود هنا = ما ينتمي لأي فئة
    '207618275541': 'Yogurts', '3451790019739': 'Yogurts', '5060337220118': 'Yogurts',
    '6281007023221': 'Yogurts', '6281018143741': 'Yogurts', '6281057005420': 'Yogurts',
    '6287004980017': 'Yogurts', '6291044444910': 'Yogurts', '8690101357343': 'Yogurts',
    '628634781029': 'Cakes & pastries', '6281100089056': 'Cakes & pastries', '9958430000002': 'Cakes & pastries',
    '6281100720157': 'Biscuits & wafers', '6291003085895': 'Biscuits & wafers',
    '859686004006': 'Chocolate',
}

def load_review(path, sample):
    r = pd.read_csv(path, dtype={'code': str})
    r['code_norm'] = r['code'].str.lstrip('0')
    r['is_correct'] = pd.to_numeric(r['is_correct'], errors='coerce')
    r = r[r['is_correct'].isin([0, 1])].copy()
    r['true_group'] = np.where(r['is_correct'] == 1, r['product_group'], r['code_norm'].map(CORRECTIONS))
    r['sample'] = sample
    return r[['code_norm', 'product_name', 'true_group', 'sample']]

if os.path.exists(GT_PATH):
    gt = pd.read_csv(GT_PATH, dtype={'code_norm': str})
else:
    gt = pd.concat([load_review(DEV_PATH, 'dev'), load_review(TEST_PATH, 'test')], ignore_index=True)
    gt = gt.drop_duplicates('code_norm')
    gt.to_csv(GT_PATH, index=False, encoding='utf-8-sig')
gt['true_group'] = gt['true_group'].replace({np.nan: None})
print(f'مجموعة التقييم: {len(gt)} منتج', gt['sample'].value_counts().to_dict())

def evaluate(expr, truth):
    """نطبق قواعد التصنيف على منتجات مجموعة التقييم ونقارن بالإجابة الصحيحة."""
    pred = (clean.withColumn('code_norm', F.regexp_replace('code', '^0+', ''))
                 .filter(F.col('code_norm').isin(truth['code_norm'].tolist()))
                 .withColumn('pred_group', expr)
                 .select('code_norm', 'pred_group').toPandas())
    out = truth.merge(pred, on='code_norm', how='left')
    out['correct'] = out['pred_group'].fillna('None') == out['true_group'].fillna('None')
    return out

ev = evaluate(group_expr, gt)
for s, g in ev.groupby('sample'):
    print(f'🎯 دقة القواعد الحالية على عينة {s}: {g.correct.mean():.1%} ({g.correct.sum()} من {len(g)})')
print(f'🎯 الإجمالي: {ev.correct.mean():.1%}')
print('\nالأخطاء الباقية:')
display(ev.loc[~ev['correct'], ['sample', 'product_name', 'pred_group', 'true_group']])

### ب) عينة Holdout نهائية 🔒
عينة ثالثة **ما شفناها أبداً** وقت تعديل القواعد. هذي اللي تعطي **الرقم الحقيقي** اللي تقولينه في المقابلة.
- تُسحب مرة وحدة بس، ويُحفظ رابطها في `reports/holdout_sheet_id.txt`، عشان إذا شغلتي الخلية مرة ثانية ما تنسحب عينة جديدة.
- راجعيها في Google Sheets: اكتبي **1** إذا التصنيف صح و**0** إذا غلط، وبعدين شغّلي الخلية اللي بعدها.
- **قاعدة مهمة:** بعد ما تشوفين نتيجة الـ Holdout، لا تعدلين القواعد عشان ترفعين رقمها. لو عدلتي، تحتاجين عينة جديدة.

In [ ]:
from google.colab import auth
auth.authenticate_user()
import gspread
from google.auth import default
creds, _ = default()
gc = gspread.authorize(creds)

HOLDOUT_ID_PATH = f'{REPORTS}/holdout_sheet_id.txt'
if os.path.exists(HOLDOUT_ID_PATH):
    HOLDOUT_SHEET_ID = open(HOLDOUT_ID_PATH).read().strip()
    print('عينة الـ Holdout موجودة من قبل')
else:
    pool = (kids.withColumn('code_norm', F.regexp_replace('code', '^0+', ''))
                .filter(~F.col('code_norm').isin(gt['code_norm'].tolist())))
    counts = {r['product_group']: r['count'] for r in pool.groupBy('product_group').count().collect()}
    fractions = {g: min(1.0, 10 / c) for g, c in counts.items()}
    holdout = (pool.sampleBy('product_group', fractions, seed=2026)
                   .select('code', 'product_name', 'brands', 'categories_tags', 'product_group')
                   .toPandas())
    holdout['is_correct'] = ''
    sh = gc.create('classification_holdout')
    sh.sheet1.update(values=[holdout.columns.tolist()] + holdout.astype(str).values.tolist(), range_name='A1')
    HOLDOUT_SHEET_ID = sh.id
    open(HOLDOUT_ID_PATH, 'w').write(HOLDOUT_SHEET_ID)
    print(f'✍️ تم إنشاء عينة Holdout فيها {len(holdout)} منتج')
print('رابط المراجعة:', f'https://docs.google.com/spreadsheets/d/{HOLDOUT_SHEET_ID}')

In [ ]:
ho = pd.DataFrame(gc.open_by_key(HOLDOUT_SHEET_ID).sheet1.get_all_records())
ho['is_correct'] = pd.to_numeric(ho['is_correct'], errors='coerce')
ho_done = ho[ho['is_correct'].isin([0, 1])]

if ho_done.empty:
    print('⏳ لسا ما راجعتي عينة الـ Holdout. عبّي عمود is_correct وارجعي شغّلي هالخلية.')
else:
    ho.to_csv(f'{REPORTS}/classification_holdout.csv', index=False, encoding='utf-8-sig')
    holdout_acc = ho_done['is_correct'].mean()
    history = pd.DataFrame([
        ('v1: بحث بجزء من الكلمة',        'dev (124)',        0.750),
        ('v2: مطابقة الوسم كامل',          'dev (124)',        0.976),
        ('v2: مطابقة الوسم كامل',          'test (123)',       0.967),
        ('v3: القواعد الحالية',            'dev + test (247)', ev['correct'].mean()),
        ('v3: القواعد الحالية',            f'holdout ({len(ho_done)})', holdout_acc),
    ], columns=['rules', 'sample', 'accuracy'])
    history.to_csv(f'{REPORTS}/classification_accuracy_history.csv', index=False, encoding='utf-8-sig')
    display(history.style.format({'accuracy': '{:.1%}'}))
    print(f'\n🔒 الدقة الحقيقية (Holdout): {holdout_acc:.1%}')
    acc_by = (ho_done.groupby('product_group')['is_correct'].agg(reviewed='count', accuracy='mean')
                     .sort_values('accuracy'))
    acc_by['accuracy'] = (acc_by['accuracy'] * 100).round(0)
    display(acc_by)
    display(ho_done.loc[ho_done['is_correct'] == 0, ['product_name', 'product_group']])

## 12) إحصائيات السكر والملح لكل فئة 🔍
### قيم الملح الشاذة
في v2 طلع متوسط الملح في أغذية الرضع 3.61 جرام لكل 100 جرام، وهذا مستحيل لأكل رضع. السبب: منتج أو اثنين قيمتهم غلط.
- **القاعدة:** أي ملح **أعلى من 5 جرام لكل 100 جرام** نعتبره خطأ إدخال. حتى الشيبس المالح تقريباً 1.5–3 جرام.
- **ما نحذف المنتج**: سكره صحيح ونحتاجه. نحط قيمة الملح بس `null` في عمود جديد `salt_clean`، ونحسب الإحصائيات منه.
- هذا اسمه **Outlier Handling** بقاعدة مبنية على فهم الموضوع (Domain Rule).

### ليش الوسيط؟
المتوسط ممكن يضلل، لأن منتجات قليلة سكرها عالي جداً ترفعه. الوسيط ما يتأثر بالقيم المتطرفة.

In [ ]:
SALT_MAX = 5.0   # جرام لكل 100 جرام

kids = (kids
        .withColumn('salt_outlier', F.coalesce(F.col('salt_100g') > SALT_MAX, F.lit(False)))
        .withColumn('salt_clean', F.when(F.col('salt_100g') <= SALT_MAX, F.col('salt_100g'))))

print('قيم ملح شاذة:', kids.filter('salt_outlier').count())
kids.filter('salt_outlier').select('product_name', 'product_group', 'salt_100g').show(truncate=40)

summary = (kids.groupBy('audience', 'product_group')
           .agg(F.count('*').alias('products'),
                F.round(F.avg('sugars_100g'), 1).alias('avg_sugar_g'),
                F.round(F.percentile_approx('sugars_100g', 0.5), 1).alias('median_sugar_g'),
                F.round(F.min('sugars_100g'), 1).alias('min_sugar_g'),
                F.round(F.max('sugars_100g'), 1).alias('max_sugar_g'),
                F.round(F.percentile_approx('sugars_100g', 0.5) / 4, 1).alias('median_sugar_tsp'),
                F.round(F.avg('salt_clean'), 2).alias('avg_salt_g'),
                F.round(F.percentile_approx('salt_clean', 0.5), 2).alias('median_salt_g'),
                F.round(F.avg(F.col('salt_clean').isNull().cast('int')) * 100, 1).alias('salt_missing_pct'))
           .orderBy(F.desc('median_sugar_g')))
summary.show(truncate=False)

import matplotlib.pyplot as plt
pdf = summary.toPandas().sort_values('median_sugar_tsp')   # ~12 صف فقط
plt.figure(figsize=(9, 6))
plt.barh(pdf['product_group'], pdf['median_sugar_tsp'], color='#e07a5f')
plt.xlabel('Teaspoons of sugar per 100g (median)')
plt.title('Sugar in products kids eat — Gulf market')
for i, v in enumerate(pdf['median_sugar_tsp']):
    plt.text(v + 0.1, i, str(v), va='center')
plt.tight_layout(); plt.show()

### أكثر 15 منتج فيه سكر، وتوزيع Nutri-Score

In [ ]:
(kids.select('product_name', 'brands', 'product_group', 'sugars_100g', 'sugar_tsp_100g', 'nutriscore_grade')
 .orderBy(F.desc('sugars_100g'))
 .show(15, truncate=40))

(kids.filter(F.col('nutriscore_grade').isin('a', 'b', 'c', 'd', 'e'))
 .groupBy('nutriscore_grade').count()
 .orderBy('nutriscore_grade')
 .show())

## 13) اختبارات البيانات التلقائية 🧪\nقبل ما نحفظ النسخة النهائية، نتأكد إن البيانات تحقق شروطنا. إذا فشل أي اختبار، الخلية توقف.

In [ ]:
results = []
def check(name, passed):
    results.append((name, passed))
    print('✅' if passed else '❌', name)

required = ['code', 'product_name', 'product_group', 'audience', 'sugars_100g', 'sugar_tsp_100g', 'salt_clean']
check('الأعمدة الأساسية موجودة', set(required) <= set(kids.columns))
check('لا يوجد باركود فاضي', kids.filter(is_blank('code')).count() == 0)
check('لا يوجد باركود مكرر', kids.count() == kids.select('code').distinct().count())
check('لا يوجد اسم فاضي', kids.filter(is_blank('product_name')).count() == 0)
check('السكر كله بين 0 و100', kids.filter(~F.col('sugars_100g').between(0, 100)).count() == 0)
check(f'الملح النظيف ما يتعدى {SALT_MAX} جرام', kids.filter(F.col('salt_clean') > SALT_MAX).count() == 0)
check('كل منتج له فئة', kids.filter(F.col('product_group').isNull()).count() == 0)
check('دقة التصنيف على مجموعة التقييم أعلى من 90%', ev['correct'].mean() > 0.90)

assert all(ok for _, ok in results), 'فيه اختبار فشل! راجعي الخطوات اللي قبل'
print('\n🎉 كل الاختبارات نجحت')

## 14) الحفظ في طبقة Gold 💾
- **Parquet:** للـ Notebook الجاي (التحليل ونظام البدائل).
- **CSV:** نسخة تنفتح في Excel أو أي أداة تحليل. نكتبه بـ Spark مباشرة، و`coalesce(1)` تجمعه في ملف واحد لأن الحجم صغير.

In [ ]:
kids.write.mode('overwrite').parquet(f'{GOLD}/kids_products.parquet')

def write_single_csv(sdf, path):
    """Spark يكتب مجلد فيه ملفات part-*. هنا نجمعها في ملف CSV واحد باسم واضح."""
    tmp = path + '_tmp'
    (sdf.coalesce(1).write.mode('overwrite')
        .option('header', True)
        .option('escape', '"')        # علامات التنصيص بالطريقة اللي يفهمها Excel وPower BI
        .csv(tmp))
    part = glob.glob(f'{tmp}/part-*.csv')[0]
    if os.path.exists(path):
        os.remove(path)
    shutil.move(part, path)
    shutil.rmtree(tmp)

write_single_csv(kids, f'{GOLD}/kids_products.csv')
write_single_csv(summary, f'{GOLD}/category_summary.csv')

for f in sorted(glob.glob(f'{PROJECT_DIR}/*/*')):
    print(f.replace(PROJECT_DIR + '/', ''))

## ✅ خلصنا Notebook 1!
**المسار كامل:** Ingestion ← Data Quality ← Cleaning ← Classification ← Evaluation (Ground Truth + Holdout) ← Outlier Handling ← EDA ← Tests ← Export


**الجاي:** Notebook 2 (التحليل بـ Spark SQL + نظام البدائل).